<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# Image Embeddings and UMAP Generation

This notebook demonstrates how to:
1. Generate vision embeddings using the [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)
2. Create a 3D UMAP visualization of the embeddings
3. Save the results for VEST

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO
import torch

In [2]:
import transformers
transformers.__version__

'4.44.1'

In [3]:
base_dir = Path("./")
images_dir = base_dir / "images"

## Load Vision Embedding Model

In [4]:
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import torch
from PIL import Image
import requests
import os

# Initialize models
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

C:\Users\rober\miniconda3\envs\genai-gpu2\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Generate Vision Embeddings for All Images

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# Get list of all image files in the images folder and subfolders
image_files = []
for root, dirs, files in os.walk(images_dir):
    for f in files:
        if f.lower().endswith('.png') or f.lower().endswith('.jpg'):
            image_files.append(os.path.join(root, f))

# Initialize lists to store results
filenames = []
embeddings = []
images = []

print(f"{len(image_files)} images found")

9285 images found


In [6]:
image_files = random.sample(image_files, 1000)

In [7]:
print(f"Processing {len(image_files)} images for embeddings...")

# Loop through all image files
for i, image_path in enumerate(image_files):
    # Obtain filename from path
    filename = os.path.relpath(image_path, images_dir)
    
    # Load the image
    current_image = Image.open(image_path)
    images.append(np.asarray(current_image))

    # Apply the processing pipeline
    current_inputs = clip_processor(images=current_image, return_tensors="pt")
    with torch.no_grad():
        current_np_emb = clip_model.get_image_features(**current_inputs).squeeze().tolist()

    # Store results
    filenames.append(filename)
    embeddings.append(current_np_emb)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(image_files)} images")



# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 1000 images for embeddings...
Processed 100/1000 images
Processed 200/1000 images
Processed 300/1000 images
Processed 400/1000 images
Processed 500/1000 images
Processed 600/1000 images
Processed 700/1000 images
Processed 800/1000 images
Processed 900/1000 images
Processed 1000/1000 images
Successfully processed 1000 images


,filename,embedding
0,test\Image_420.jpg,"[0.18075445294380188, -0.13964124023914337, -0..."
1,train\Image_420.jpg,"[0.144201397895813, -0.2779114842414856, -0.11..."
2,test\Image_2562.jpg,"[0.2095768302679062, -0.04088331013917923, -0...."
3,test\Image_567.jpg,"[0.04608976095914841, -0.10046644508838654, -0..."
4,test\Image_30.jpg,"[0.21522346138954163, 0.060987163335084915, -0..."


## Create 3D UMAP Visualization from Embeddings

In [8]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (1000, 512)
Creating 3D UMAP coordinates...


C:\Users\rober\miniconda3\envs\genai-gpu2\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: 10.37 to 16.81
Y range: 0.81 to 8.06
Z range: -2.44 to 3.13


,filename,embedding,x,y,z
0,test\Image_420.jpg,"[0.18075445294380188, -0.13964124023914337, -0...",11.708443,5.975971,1.160136
1,train\Image_420.jpg,"[0.144201397895813, -0.2779114842414856, -0.11...",14.487134,3.430255,-2.071525
2,test\Image_2562.jpg,"[0.2095768302679062, -0.04088331013917923, -0....",12.263655,5.897109,-0.066880
3,test\Image_567.jpg,"[0.04608976095914841, -0.10046644508838654, -0...",14.091143,7.067309,-0.240073
4,test\Image_30.jpg,"[0.21522346138954163, 0.060987163335084915, -0...",14.086477,7.658654,-0.065490


## Save Results to CSV File

In [9]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (1000, 5)
Columns: ['filename', 'embedding', 'x', 'y', 'z']


,filename,x,y,z
0,test\Image_420.jpg,11.708443,5.975971,1.160136
1,train\Image_420.jpg,14.487134,3.430255,-2.071525
2,test\Image_2562.jpg,12.263655,5.897109,-0.066880
3,test\Image_567.jpg,14.091143,7.067309,-0.240073
4,test\Image_30.jpg,14.086477,7.658654,-0.065490
5,train\Image_5494.jpg,14.376655,4.078625,0.102398
6,test\Image_936.jpg,14.032869,4.623199,0.373962
7,train\Image_2787.jpg,11.800838,5.943596,0.916217
8,test\Image_1359.jpg,13.118350,7.864383,-0.770244
9,test\Image_1026.jpg,13.587582,7.640502,-0.290553
